# Time discretization
\
CADET uses IDAS from the [<u>SUNDIALS software package</u>](https://sundials.readthedocs.io) to integrate the spatially semi-discretized equations in time.
Furthermore, IDAS can handle ODAE and parameter sensitivities.
ODAE are used for rapid-equilibrium adsorption and reactions, refer e.g. to the [<u>Equation-Generator</u>](https://cadet-equations-74ko8eryoxmsbqspggxrj2.streamlit.app/).
\
IDAS uses a numerical technique called a Backward Differentiation Formula (BDF) to advance the solution in time.
This method is an implicit time integration scheme and thus well-suited for stiff systems – including situations where different parts of the system evolve at very different speeds, which is something that often appears in chromatography models (see the later section on stiffness).

Because the BDF method is implicit, each time step requires solving a nonlinear system of algebraic equation that comes from the spatially discretized model.
That is, IDAS must find a new solution value 
The BDF gives a nonlinear algebraic system to be solved at each time step\
$$
G(y_n) := F \left( t_n, y_n, h^{1}_n \sum_{i=0}^q \alpha_n y_{n-i} \right) = 0,
$$
where $F$ is the **residual** function defined by the spatially discretized model equations, $t_n, y_n, h_n$ are the current time point, solution, and time step, $\alpha_{n,i}$ is a factor depending on the time step size $h$ and solution index $i$.
IDAS determines the time step size, based on the solution trajectories and error tolerances suing a predictor, corrector approach.

The non-linear system is solved using a [<u>Newton method</u>](https://en.wikipedia.org/wiki/Newton%27s_method).

### CADET user and developer interaction with IDAS:

- Specifying error tolerances: how accurate does our solution need to be? Trade-off between computational demands and accuracy

- Providing a Jacobian to solve

$$
\frac{\partial G}{\partial y} \left[y_{n,m+1} - y_{n,m} \right] = -G(y_{n,m}),
$$

where $y_{n,m}$ is the current solution of the $m$th Newton iteration.
That is, CADET-Core requires some approximation of $\frac{\partial G}{\partial y}$ to be handed to IDAS.
To this end, we can either derive the Jacobian analytically and implement it, or rely on Algorithmic Differentiation to compute the Jacobian (which requires additional computations and is thus slower).
This must be done when expanding CADET-Core by e.g. a new reaction or adsorption model, numerical discretization method, transport model, etc.

- Providing consistent initial values: The BDF solves an initial value problems (IVP), meaning we need to provide initial values that are consistent with the model equation.
That is, the equation must hold when the initial values are inserted.


### Question:
#### How does the solution behave with different error tolerances?
#### Are abstol and reltol independent of each other?
#### How should I select abstol and reltol for my simulation?

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, interactive
import ipywidgets as widgets
import os

from cadet import Cadet
import utility.convergence as convergence
import utility.setting_Col1D_SMA_4comp_LWE_benchmark1 as lwe

Cadet.cadet_path = r"C:\Users\jmbr\OneDrive\Desktop\CADET_compiled\master5_generalizedUnit_f1a1972\aRELEASE"
path = os.getcwd()

def graph_column(idas_abstol=1e-1, idas_reltol=1e-1):

    model = Cadet()
    model.root = lwe.get_model(
        spatial_method_bulk=0,
        spatial_method_particle=0,
        particle_type='GENERAL_RATE_PARTICLE',
        axRefinement=2,
        parZ=4,
        return_bulk=True,
        idas_abstol=idas_abstol,
        idas_reltol=idas_reltol
    )
    
    model.filename = 'test.h5'
    model.save()
    model.run_simulation()

    outlet = convergence.get_outlet(path+'/'+model.filename, unit="000")
    sol_time = convergence.get_solution_times(path+'/'+model.filename)
    
    reference = convergence.get_outlet(path+'/data/ref_LWE.h5', unit="000")
    errorComp = np.max(abs(reference[:, 1:] - outlet[:, 1:])) / np.max(abs(reference[:, 1:]))
    errorSalt = np.max(abs(reference[:, 0] - outlet[:, 0])) / np.max(abs(reference[:, 0]))
    
    fig, axs = plt.subplots(1, 2, figsize=(12, 5))
    
    # First plot
    axs[0].plot(sol_time, outlet[:, 0])
    axs[0].set_xlabel(r'$x~/~M$')
    axs[0].set_ylabel(r'$concentration~/~mol \cdot M^{-3}$')
    axs[0].set_title("Salt")
    
    axs[1].plot(sol_time, outlet[:, 1:])
    axs[1].set_xlabel(r'$x~/~M$')
    axs[1].set_title("components")

    fig.suptitle(
        f"Sim. time: {convergence.get_compute_time(path + '/' + model.filename):.3e}" + f", rel. max. error salt: {errorSalt*100:.2f}%," + f" rel. max. error comps: {errorComp*100:.2f}%",
        fontsize=14
    )
    
    plt.tight_layout()
    plt.show()

abstol_steps = [10**i for i in range(-12, 3)]
abstol_options = [(f"{v:.0e}", v) for v in abstol_steps]

reltol_steps = [10**i for i in range(-12, 3)]
reltol_options = [(f"{v:.0e}", v) for v in reltol_steps]

interact(
    graph_column,
    idas_abstol=widgets.SelectionSlider(
        options=abstol_options,
        description="abstol"
    ),
    idas_reltol=widgets.SelectionSlider(
        options=reltol_options,
        description="reltol"
    )
)



interactive(children=(SelectionSlider(description='abstol', options=(('1e-12', 1e-12), ('1e-11', 1e-11), ('1e-…

<function __main__.graph_column(idas_abstol=0.1, idas_reltol=0.1)>

### Discussion
Error bounds for the local truncation error test are specified by an absolute tolerance (`ABSTOL`) and a relative tolerance (`RELTOL`). Note that the relative tolerance only works for non-zero values, whereas zero values are accounted for by the absolute tolerance. For example, a relative tolerance of $10^{-4}$ and absolute tolerance of $10^{-8}$ requests $3$ significant digits (correct digits after the comma in scientific notation) and considers all numbers with magnitude smaller than $10^{-8}$ as zero.

$|error| < ABSTOL + RELTOL \cdot |y|_{max}$

Choosing a reasonable tolerance is very relevant when compute time is limiting.\
In such a case, consider the formula above to find a suitable tolerance and add a safety threshold.\
Otherwise just set the tolerances very low, e.g. $abstol = 1e-10$ and $reltol = 1e-8$.


## Stiffness and computational requirements

Stiffness is a hard to grasp concept and has many facettes.
- One important, maybe the most important, factor causing stiffness is the presence of different time scales.
This happens e.g. when mass transfer based on adsorption on one hand happens much faster than convective transport on the other.
- Another source is highly non-linearly dependent equations, e.g. highly non-linear adsorption models.
- Another source is when we don't have $u << D_{ax}$, but only $u < D_{ax}$, i.e. when bulk transport is less convection dominated

**TODO:**\
create examples that are potentially stiff, play around with the relevant parameters and see how that impacts stiffness, which we simply measure by the required simulation time.\
For an adsorption kinetics example, additionally compare to req. adsorption, does this reduce stiffness (measured by required simulation time), compared to kinetic adsorption with fast kinetics?


## Providing the Jaocbian
\
**TODO:**
what does the Jacobian look like, where are different parts of the model located?
<img src="data/img/Jacobian.png" width="350" alt="Jacobian">